# Full-throttle speed loss in 2026 F1 qualifying: season run

Runs the Japan-validated pipeline (`flatout.run_session`) on the main Qualifying session of every 2026 event (conventional and sprint weekends; Sprint Qualifying is not used). Each session's tables go to `data/season/`; the QC summary has one row per session. Validation 1 needs each circuit's official length and is skipped here (`official_length_m=None`). OpenF1 brake values come from `openf1_session_key`; without a key, brake is taken at face value and logged.

In [ ]:
import re
from pathlib import Path

import fastf1
import numpy as np
import pandas as pd

from flatout import openf1_session_key, run_session, to_seconds

YEAR = 2026
OUT = Path("data/season")
OUT.mkdir(parents=True, exist_ok=True)

schedule = fastf1.get_event_schedule(YEAR, include_testing=False)
events = schedule[schedule.EventFormat.isin(["conventional", "sprint_qualifying"])]  # every one has a main "Qualifying"
miami = events.loc[events.EventName.str.contains("Miami"), "RoundNumber"].iloc[0]  # rule changes apply from Miami onward
events = events.assign(Period=np.where(events.RoundNumber < miami, "pre_miami", "post_miami"))
events[["RoundNumber", "EventName", "Location", "EventFormat", "Period"]]

## Run (one session at a time; a failure is logged and the loop continues)

In [ ]:
rows = []
for ev in events.itertuples():
    row = dict(Round=ev.RoundNumber, Event=ev.EventName, Period=ev.Period)
    try:
        tables, qc = run_session(YEAR, ev.RoundNumber, openf1_session=openf1_session_key(YEAR, ev.RoundNumber))
        stem = OUT / f"{ev.RoundNumber:02d}_{re.sub(r'[^A-Za-z0-9]+', '_', ev.Location).strip('_').lower()}"
        to_seconds(tables["sectors"]).to_csv(f"{stem}_sectors.csv", index=False)
        to_seconds(tables["segments"]).to_csv(f"{stem}_segments.csv", index=False)
        tables["exclusions"].to_csv(f"{stem}_exclusions.csv", index=False)
        row.update(Status="ok", **qc, Exclusions=len(tables["exclusions"]),
                   CleanPerStraight=tables["coverage"].loc["straight"].to_dict())
    except Exception as e:
        row.update(Status=f"error: {type(e).__name__}: {e}")
    rows.append(row)
season_qc = pd.DataFrame(rows)
season_qc.to_csv(OUT / "qc_summary.csv", index=False)

## QC summary (one row per session)

In [ ]:
cols = ["Round", "Event", "Period", "Status", "brake_source", "push_laps", "push_lap_frozen_pct_mean",
        "CleanPerStraight", "Exclusions"]
season_qc.reindex(columns=cols).rename(columns={"push_lap_frozen_pct_mean": "FreezeShare_pct"}).round(1)